[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

# ONNX Proto Structure — Hands-On Application

| # | Section | Description |
|---|---------|-------------|
| 1 | [Setup](#1-setup) | Imports and helpers |
| 2 | [Exercise 1: Model Anatomy Inspector](#2-exercise-1) | Traverse and report all proto fields |
| 3 | [Exercise 2: Graph Topology Analysis](#3-exercise-2) | Compute in/out degree, find bottlenecks |
| 4 | [Exercise 3: Build Multi-Branch Model](#4-exercise-3) | ResNet-style skip connections |
| 5 | [Exercise 4: Modify Proto Fields](#5-exercise-4) | Programmatically edit a model |
| 6 | [Exercise 5: Weight Statistics](#6-exercise-5) | Analyze initializer distributions |
| 7 | [Exercise 6: Graph Visualization](#7-exercise-6) | NetworkX-based graph renderer |
| 8 | [Challenge: Model Diff Tool](#8-challenge) | Compare two model proto structures |
| 9 | [Summary](#9-summary) | Skills review |

In [ ]:
# !pip install onnx numpy matplotlib networkx --quiet

import onnx
from onnx import helper, TensorProto, checker, numpy_helper, AttributeProto
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
# Helper: build a reusable two-layer MLP model
def build_mlp(hidden_dim=64, out_dim=10, input_dim=784):
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", input_dim])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", out_dim])
    W1 = numpy_helper.from_array(np.random.randn(input_dim, hidden_dim).astype(np.float32)*0.01, "W1")
    b1 = numpy_helper.from_array(np.zeros(hidden_dim, dtype=np.float32), "b1")
    W2 = numpy_helper.from_array(np.random.randn(hidden_dim, out_dim).astype(np.float32)*0.01, "W2")
    b2 = numpy_helper.from_array(np.zeros(out_dim, dtype=np.float32), "b2")
    nodes = [
        helper.make_node("MatMul", ["X", "W1"], ["h1"]),
        helper.make_node("Add", ["h1", "b1"], ["z1"]),
        helper.make_node("Relu", ["z1"], ["a1"]),
        helper.make_node("MatMul", ["a1", "W2"], ["h2"]),
        helper.make_node("Add", ["h2", "b2"], ["Y"]),
    ]
    graph = helper.make_graph(nodes, "mlp", [X], [Y], initializer=[W1, b1, W2, b2])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model.producer_name = "exercise"
    checker.check_model(model)
    return model

mlp = build_mlp()
print(f"MLP model: {len(mlp.graph.node)} nodes, {len(mlp.graph.initializer)} weights")

## 2. Exercise 1: Model Anatomy Inspector <a id="2-exercise-1"></a>

Write a function that traverses a `ModelProto` and produces a comprehensive report of its structure. The report should cover every level of the hierarchy:

$$\texttt{ModelProto} \to \texttt{GraphProto} \to \{\texttt{NodeProto}^*, \texttt{TensorProto}^*, \texttt{ValueInfoProto}^*\}$$

**Requirements:**
1. Print ModelProto-level metadata
2. Count and categorize nodes by op_type
3. Summarize initializer sizes and total parameter count
4. List input/output shapes

In [ ]:
def model_report(model: onnx.ModelProto) -> dict:
    """Generate a comprehensive report of model structure."""
    g = model.graph
    
    # Op counts
    op_counts = {}
    for n in g.node:
        op_counts[n.op_type] = op_counts.get(n.op_type, 0) + 1
    
    # Initializer stats
    total_params = 0
    total_bytes = 0
    init_info = []
    for init in g.initializer:
        dims = list(init.dims)
        n_elements = int(np.prod(dims)) if dims else 0
        raw_len = len(init.raw_data) if init.raw_data else 0
        total_params += n_elements
        total_bytes += raw_len
        init_info.append((init.name, dims, n_elements, raw_len))
    
    # Print report
    print("╔══════════════════════════════════════════════╗")
    print("║         ONNX Model Anatomy Report            ║")
    print("╠══════════════════════════════════════════════╣")
    print(f"║  IR Version:       {model.ir_version:<25} ║")
    print(f"║  Producer:         {model.producer_name:<25} ║")
    print(f"║  Graph name:       {g.name:<25} ║")
    print(f"║  Total nodes:      {len(g.node):<25} ║")
    print(f"║  Total params:     {total_params:<25,} ║")
    print(f"║  Weight bytes:     {total_bytes:<25,} ║")
    print(f"║  Serialized size:  {len(model.SerializeToString()):<25,} ║")
    print("╠══════════════════════════════════════════════╣")
    print("║  OpSet Imports:")
    for oi in model.opset_import:
        d = oi.domain or '(default)'
        print(f"║    {d}: v{oi.version}")
    print("║")
    print("║  Operator Counts:")
    for op, count in sorted(op_counts.items(), key=lambda x: -x[1]):
        bar = '█' * count
        print(f"║    {op:<15} {count:>3} {bar}")
    print("║")
    print("║  Initializers:")
    for name, dims, n_elem, nbytes in init_info:
        print(f"║    {name:<20} {str(dims):<20} {n_elem:>8,} params  {nbytes:>8,} bytes")
    print("╚══════════════════════════════════════════════╝")
    
    return {'op_counts': op_counts, 'total_params': total_params, 'total_bytes': total_bytes}

report = model_report(mlp)

## 3. Exercise 2: Graph Topology Analysis <a id="3-exercise-2"></a>

Analyze the graph's topology by computing node connectivity metrics. For each node $n_i$, compute:
- **In-degree**: number of distinct source nodes (producers of its inputs)
- **Out-degree**: number of distinct consumer nodes (consumers of its outputs)

Nodes with high fan-out are potential parallelism points; nodes with high fan-in are synchronization points.

In [ ]:
def analyze_topology(model: onnx.ModelProto):
    """Analyze graph topology: connectivity, depth, width."""
    g = model.graph
    
    # Map: tensor_name -> producer node index
    producers = {}
    for inp in g.input:
        producers[inp.name] = 'input'
    for init in g.initializer:
        producers[init.name] = 'init'
    for i, n in enumerate(g.node):
        for out in n.output:
            producers[out] = i
    
    # Compute in/out degree for each node
    consumers = {i: set() for i in range(len(g.node))}
    in_degree = {i: 0 for i in range(len(g.node))}
    
    for j, n in enumerate(g.node):
        for inp_name in n.input:
            if inp_name and inp_name in producers:
                src = producers[inp_name]
                if isinstance(src, int):
                    consumers[src].add(j)
                    in_degree[j] += 1
    
    print(f"{'Idx':>3} │ {'Op':<15} │ {'In-deg':>6} │ {'Out-deg':>7} │ {'Inputs':<30} │ Outputs")
    print("─" * 90)
    for i, n in enumerate(g.node):
        out_deg = len(consumers[i])
        in_deg = in_degree[i]
        print(f"{i:>3} │ {n.op_type:<15} │ {in_deg:>6} │ {out_deg:>7} │ {str(list(n.input)):<30} │ {list(n.output)}")
    
    # Graph depth (longest path)
    depth = {}
    def get_depth(idx):
        if idx in depth:
            return depth[idx]
        max_parent = 0
        for inp_name in g.node[idx].input:
            if inp_name and inp_name in producers:
                src = producers[inp_name]
                if isinstance(src, int):
                    max_parent = max(max_parent, get_depth(src) + 1)
        depth[idx] = max_parent
        return max_parent
    
    for i in range(len(g.node)):
        get_depth(i)
    
    max_depth = max(depth.values()) if depth else 0
    print(f"\nGraph depth (longest path): {max_depth + 1} levels")
    print(f"Total edges: {sum(len(c) for c in consumers.values())}")

analyze_topology(mlp)

## 4. Exercise 3: Build Multi-Branch Model <a id="4-exercise-3"></a>

Build a model with **skip connections** (like ResNet). This requires the graph to have a multi-branch topology:

```
X ──┬── Linear ── Relu ── Linear ──┬── Add ── Y
    │                               │
    └───────── (skip connection) ───┘
```

The residual connection computes: $Y = F(X) + X$ where $F$ is the two-layer transform.

In [ ]:
dim = 32

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", dim])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", dim])

W1 = numpy_helper.from_array(np.random.randn(dim, dim).astype(np.float32) * 0.01, "W1")
b1 = numpy_helper.from_array(np.zeros(dim, dtype=np.float32), "b1")
W2 = numpy_helper.from_array(np.random.randn(dim, dim).astype(np.float32) * 0.01, "W2")
b2 = numpy_helper.from_array(np.zeros(dim, dtype=np.float32), "b2")

nodes = [
    helper.make_node("MatMul", ["X", "W1"], ["h1"], name="branch_mm1"),
    helper.make_node("Add", ["h1", "b1"], ["z1"], name="branch_add1"),
    helper.make_node("Relu", ["z1"], ["a1"], name="branch_relu"),
    helper.make_node("MatMul", ["a1", "W2"], ["h2"], name="branch_mm2"),
    helper.make_node("Add", ["h2", "b2"], ["z2"], name="branch_add2"),
    # Skip connection: Y = z2 + X
    helper.make_node("Add", ["z2", "X"], ["Y"], name="residual_add"),
]

graph = helper.make_graph(nodes, "residual_block", [X], [Y],
                          initializer=[W1, b1, W2, b2])
res_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
checker.check_model(res_model)

print("Residual block model built!")
analyze_topology(res_model)

# Verify: output should be close to input (since weights are near zero)
try:
    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(res_model)
    x = np.random.randn(4, dim).astype(np.float32)
    y = ev.run(None, {"X": x})[0]
    print(f"\nInput norm:  {np.linalg.norm(x):.4f}")
    print(f"Output norm: {np.linalg.norm(y):.4f}")
    print(f"Residual (Y-X) norm: {np.linalg.norm(y - x):.6f} (small because W ≈ 0)")
except Exception as e:
    print(f"Eval: {e}")

## 5. Exercise 4: Modify Proto Fields <a id="5-exercise-4"></a>

ONNX models are mutable Protobuf objects — you can programmatically modify fields after construction. This is useful for:
- Adding/changing metadata
- Renaming tensors
- Swapping initializer values (e.g., loading different weights)
- Adjusting operator attributes

**Task:** Take the MLP model and modify it in several ways, validating after each change.

In [ ]:
import copy

# Work on a copy
modified = copy.deepcopy(mlp)

# 1. Update metadata
modified.producer_name = "modified_exercise"
modified.doc_string = "MLP with modified metadata and weights"
entry = modified.metadata_props.add()
entry.key = "modified_by"
entry.value = "proto_structure_exercise"

# 2. Replace W1 with identity-like matrix (for testing)
for i, init in enumerate(modified.graph.initializer):
    if init.name == "W1":
        dims = list(init.dims)
        min_dim = min(dims)
        identity_like = np.eye(dims[0], dims[1], dtype=np.float32) * 0.1
        new_init = numpy_helper.from_array(identity_like, name="W1")
        modified.graph.initializer[i].CopyFrom(new_init)
        print(f"Replaced W1 with scaled identity ({dims[0]}x{dims[1]})")

# 3. Add a Softmax at the end
old_output_name = modified.graph.node[-1].output[0]
modified.graph.node[-1].output[0] = "pre_softmax"
softmax_node = helper.make_node("Softmax", ["pre_softmax"], ["Y"], axis=1)
modified.graph.node.append(softmax_node)

checker.check_model(modified)
print(f"\nModified model is valid!")
print(f"  Nodes: {[n.op_type for n in modified.graph.node]}")
print(f"  Producer: {modified.producer_name}")
print(f"  Metadata: {[(p.key, p.value) for p in modified.metadata_props]}")

## 6. Exercise 5: Weight Statistics <a id="6-exercise-5"></a>

Analyze the distribution of weights across all initializers. For each weight tensor $W$, compute:

- Mean: $\mu = \frac{1}{N} \sum_{i=1}^{N} w_i$
- Std: $\sigma = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (w_i - \mu)^2}$
- Sparsity: $\text{sparsity} = \frac{|\{i : |w_i| < \epsilon\}|}{N}$

In [ ]:
def weight_analysis(model: onnx.ModelProto, eps: float = 1e-6):
    """Analyze weight distributions for all initializers."""
    float_inits = []
    for init in model.graph.initializer:
        arr = numpy_helper.to_array(init)
        if arr.dtype in (np.float32, np.float64):
            float_inits.append((init.name, arr))
    
    if not float_inits:
        print("No float initializers found.")
        return
    
    print(f"{'Name':<12} │ {'Shape':<15} │ {'Mean':>10} │ {'Std':>10} │ {'Min':>10} │ {'Max':>10} │ {'Sparsity':>8}")
    print("─" * 90)
    for name, arr in float_inits:
        flat = arr.ravel()
        sparsity = np.mean(np.abs(flat) < eps) * 100
        print(f"{name:<12} │ {str(arr.shape):<15} │ {flat.mean():>10.6f} │ {flat.std():>10.6f} │ {flat.min():>10.6f} │ {flat.max():>10.6f} │ {sparsity:>7.1f}%")
    
    # Combined histogram
    fig, axes = plt.subplots(1, len(float_inits), figsize=(4*len(float_inits), 4))
    if len(float_inits) == 1:
        axes = [axes]
    
    for ax, (name, arr) in zip(axes, float_inits):
        flat = arr.ravel()
        ax.hist(flat, bins=50, color='#2196F3', alpha=0.7, edgecolor='white')
        ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
        ax.set_title(f"{name} ({arr.shape})", fontsize=10, fontweight='bold')
        ax.set_xlabel('Value')
        ax.set_ylabel('Count')
    
    plt.suptitle('Weight Distributions', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

weight_analysis(mlp)

## 7. Exercise 6: Graph Visualization <a id="7-exercise-6"></a>

Build a reusable graph visualizer that renders the ONNX computation graph using NetworkX and matplotlib.

In [ ]:
def visualize_onnx_graph(model: onnx.ModelProto, title: str = "ONNX Graph"):
    g = model.graph
    G = nx.DiGraph()
    init_names = {i.name for i in g.initializer}
    output_names = {o.name for o in g.output}
    
    for inp in g.input:
        ntype = 'weight' if inp.name in init_names else 'input'
        G.add_node(inp.name, ntype=ntype)
    
    for i, node in enumerate(g.node):
        nid = f"{node.op_type}_{i}"
        G.add_node(nid, ntype='op')
        for inp in node.input:
            if inp:
                if inp not in G:
                    G.add_node(inp, ntype='tensor')
                G.add_edge(inp, nid)
        for out in node.output:
            nt = 'output' if out in output_names else 'tensor'
            G.add_node(out, ntype=nt)
            G.add_edge(nid, out)
    
    color_map = {'input': '#4CAF50', 'weight': '#FF9800', 'op': '#2196F3',
                 'tensor': '#BDBDBD', 'output': '#F44336'}
    size_map = {'input': 1500, 'weight': 800, 'op': 2000, 'tensor': 600, 'output': 1500}
    
    colors = [color_map.get(G.nodes[n].get('ntype', 'tensor'), '#BDBDBD') for n in G]
    sizes = [size_map.get(G.nodes[n].get('ntype', 'tensor'), 600) for n in G]
    labels = {}
    for n in G:
        nt = G.nodes[n].get('ntype', '')
        if nt == 'op':
            labels[n] = n.rsplit('_', 1)[0]
        elif nt in ('input', 'output'):
            labels[n] = n
        else:
            labels[n] = n if len(n) < 8 else n[:6] + '..' 
    
    fig, ax = plt.subplots(figsize=(12, 8))
    pos = nx.kamada_kawai_layout(G)
    nx.draw(G, pos, ax=ax, with_labels=True, labels=labels,
            node_color=colors, node_size=sizes, font_size=7,
            font_weight='bold', arrows=True, arrowsize=15,
            edge_color='#78909C', width=1.5, alpha=0.9)
    
    legend_elems = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c,
                               markersize=10, label=t.title())
                    for t, c in color_map.items()]
    ax.legend(handles=legend_elems, loc='upper left')
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_onnx_graph(res_model, "Residual Block — Graph Topology")

## 8. Challenge: Model Diff Tool <a id="8-challenge"></a>

Build a tool that compares two `ModelProto` instances and reports differences at every level of the hierarchy. This is useful for debugging export issues or verifying optimization passes.

**Comparison dimensions:**
- Metadata (producer, IR version, opsets)
- Graph structure (node count, op types)
- Weight sizes and value changes

In [ ]:
def model_diff(m1: onnx.ModelProto, m2: onnx.ModelProto,
               name1: str = "Model A", name2: str = "Model B"):
    """Compare two ModelProto instances and report differences."""
    print(f"\n{'═' * 60}")
    print(f"  Model Diff: {name1} vs {name2}")
    print(f"{'═' * 60}")
    
    # Metadata
    meta_fields = ['ir_version', 'producer_name', 'producer_version', 'domain', 'model_version']
    print("\n  Metadata:")
    for field in meta_fields:
        v1 = getattr(m1, field)
        v2 = getattr(m2, field)
        marker = '≠' if v1 != v2 else '='
        print(f"    {field:<20} {marker} {v1!r:<20} vs {v2!r}")
    
    # Graph structure
    g1, g2 = m1.graph, m2.graph
    print(f"\n  Graph Structure:")
    print(f"    Nodes:        {len(g1.node):>4} vs {len(g2.node):>4}")
    print(f"    Inputs:       {len(g1.input):>4} vs {len(g2.input):>4}")
    print(f"    Outputs:      {len(g1.output):>4} vs {len(g2.output):>4}")
    print(f"    Initializers: {len(g1.initializer):>4} vs {len(g2.initializer):>4}")
    
    ops1 = [n.op_type for n in g1.node]
    ops2 = [n.op_type for n in g2.node]
    if ops1 != ops2:
        print(f"    Op sequence differs!")
        print(f"      {name1}: {ops1}")
        print(f"      {name2}: {ops2}")
    else:
        print(f"    Op sequence: identical")
    
    # Size comparison
    s1 = len(m1.SerializeToString())
    s2 = len(m2.SerializeToString())
    print(f"\n  Serialized Size:")
    print(f"    {name1}: {s1:,} bytes")
    print(f"    {name2}: {s2:,} bytes")
    print(f"    Delta:  {s2-s1:+,} bytes ({(s2-s1)/s1*100:+.1f}%)")
    print(f"{'═' * 60}")

model_diff(mlp, modified, "Original MLP", "Modified MLP")

In [ ]:
# Visualize the size breakdown of the MLP model
g = mlp.graph
total = len(mlp.SerializeToString())

weight_bytes = sum(len(init.raw_data) for init in g.initializer if init.raw_data)
graph_meta = total - weight_bytes

init_sizes = [(init.name, len(init.raw_data) if init.raw_data else 0)
              for init in g.initializer]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.pie([weight_bytes, graph_meta], labels=['Weight Data', 'Graph Metadata'],
       autopct='%1.1f%%', colors=['#2196F3', '#FF9800'], startangle=90)
ax.set_title(f'Model Size Breakdown ({total:,} bytes)', fontweight='bold')

ax = axes[1]
names = [n for n, s in init_sizes if s > 0]
sizes = [s for n, s in init_sizes if s > 0]
ax.barh(names, [s/1024 for s in sizes], color='#4CAF50')
ax.set_xlabel('Size (KB)')
ax.set_title('Initializer Sizes', fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 9. Summary

In this hands-on notebook you have:

1. **Built a model anatomy inspector** that traverses the full `ModelProto → GraphProto → NodeProto/TensorProto/ValueInfoProto` hierarchy

2. **Analyzed graph topology** — computed in/out degree, graph depth, and identified parallelism and synchronization points

3. **Constructed a multi-branch model** with skip connections, demonstrating that ONNX graphs are DAGs, not linear chains

4. **Modified proto fields programmatically** — updated metadata, replaced weights, and added new nodes to an existing model

5. **Analyzed weight distributions** with statistics ($\mu$, $\sigma$, sparsity) and histograms

6. **Visualized computation graphs** using NetworkX with color-coded node types

7. **Built a model diff tool** for comparing two ModelProto instances at every hierarchy level